In [1]:
import pandas as pd
import numpy as np
from functions import scrape_nba_data
import matplotlib.pyplot as plt

In [ ]:
raw_df = scrape_nba_data(2021, 2026)
raw_df.head()

In [ ]:
raw_df.describe()

In [ ]:
raw_df.dtypes

In [ ]:
# Some players get traded mid-season: the API gives one row per team plus a
# "TOT" (total) row combining them. Keep only the TOT row when present so
# each player has exactly one row per season. (A groupby(...).apply() here
# would silently drop PLAYER_ID/SEASON under current pandas, since group-by
# columns are excluded from what's passed to the function — so this uses a
# vectorized transform instead.)
has_tot = raw_df.groupby(["PLAYER_ID", "SEASON"])["TEAM_ABBREVIATION"].transform(
    lambda s: (s == "TOT").any()
)
df = raw_df[~has_tot | (raw_df["TEAM_ABBREVIATION"] == "TOT")].reset_index(drop=True)

# eda_box_scores.py expects the scraper's original column names (SEASON,
# PLAYER_ID, FG_PCT, GP, MIN, FGA, ...), so the deduplicated data is saved
# under those names — this is the file the EDA cell below reads.
df.to_csv("nba_data.csv", index=False)

# A friendlier, renamed per-game table for display/export. Not used by the
# EDA script, which relies on the original column names saved above.
id_cols = ["SEASON", "PLAYER_NAME", "TEAM_ABBREVIATION", "AGE", "PLAYER_POSITION"]

shooting_cols = ["FG_PCT", "FG3_PCT", "FT_PCT"]

per_game_cols = ["PTS", "REB", "AST", "STL", "TOV"]

advanced_cols = [
    "OFF_RATING", "DEF_RATING", "NET_RATING",
    "TS_PCT", "EFG_PCT", "USG_PCT", "AST_PCT", "REB_PCT", "PIE", "PACE",
]

keep_cols = id_cols + shooting_cols + per_game_cols + advanced_cols
keep_cols = [c for c in keep_cols if c in df.columns]

per_game = df[keep_cols].rename(columns={
    "SEASON": "YEAR",
    "PLAYER_NAME": "PLAYER",
    "TEAM_ABBREVIATION": "TEAM",
    "PTS": "PPG",
    "REB": "RPG",
    "AST": "APG",
    "STL": "SPG",
    "TOV": "TOPG",
})

per_game = per_game.sort_values(["YEAR", "PLAYER"]).reset_index(drop=True)

per_game.to_csv("nba_per_game.csv", index=False)
per_game.head()


In [ ]:
from eda_box_scores import run_eda

run_eda(input_csv="nba_data.csv", output_dir="eda_output")

## Bayesian Gaussian AR(1) shooting model

Projects a player's true shooting percentage (TS%) from their age, prior-season
TS%, and usage rate (USG%). The spec below is informed directly by the EDA
above rather than guessed:

- **AR(1) term** on `PREV_TS_PCT` — the lag-1 check above showed real
  season-to-season persistence in TS%, so the model should lean on it.
- **Age + age²** — the age-vs-TS% relationship above is curved, not linear.
- **Usage rate** — TS% and USG% trade off within a season (higher-usage
  players tend to take harder, lower-efficiency shots), so USG% is included
  as a covariate alongside the AR term.
- **Gaussian likelihood** — TS% is roughly bell-shaped and continuous once
  the low-attempts players are filtered out (same floor as the EDA above).

Requirements: `pip install pymc arviz`

In [ ]:
import os

import pymc as pm
import arviz as az

from eda_box_scores import load_data

MIN_SEASON_FGA = 50  # same attempts floor used in the EDA distribution/autocorrelation checks

model_source = load_data("nba_data.csv")

ar_cols = ["PLAYER_ID", "SEASON_START_YEAR", "AGE", "TS_PCT", "USG_PCT", "SEASON_FGA_EST"]
ar_df = (
    model_source[ar_cols]
    .dropna()
    .sort_values(["PLAYER_ID", "SEASON_START_YEAR"])
)
ar_df = ar_df[ar_df["SEASON_FGA_EST"] >= MIN_SEASON_FGA]

ar_df["PREV_TS_PCT"] = ar_df.groupby("PLAYER_ID")["TS_PCT"].shift(1)
ar_df["PREV_SEASON"] = ar_df.groupby("PLAYER_ID")["SEASON_START_YEAR"].shift(1)

# Only keep true consecutive-season pairs -- same rule as the EDA autocorrelation
# check, so an injury/G-League/out-of-league gap year isn't treated as a lag-1 pair.
ar_model_df = (
    ar_df[ar_df["SEASON_START_YEAR"] - ar_df["PREV_SEASON"] == 1]
    .dropna(subset=["PREV_TS_PCT"])
    .reset_index(drop=True)
)

print(f"{len(ar_model_df)} player-seasons with a valid prior season, age, and usage rate")
ar_model_df.head()

In [ ]:
# Standardize predictors so the priors below are on a comparable, weakly-informative scale
y = ar_model_df["TS_PCT"].to_numpy()
age = ar_model_df["AGE"].to_numpy()
usg = ar_model_df["USG_PCT"].to_numpy()
prev_ts = ar_model_df["PREV_TS_PCT"].to_numpy()

age_mean, age_sd = age.mean(), age.std()
usg_mean, usg_sd = usg.mean(), usg.std()
prev_mean, prev_sd = prev_ts.mean(), prev_ts.std()

age_z = (age - age_mean) / age_sd
usg_z = (usg - usg_mean) / usg_sd
prev_z = (prev_ts - prev_mean) / prev_sd

In [ ]:
with pm.Model() as ts_ar_model:
    alpha = pm.Normal("alpha", mu=y.mean(), sigma=0.2)
    phi_prev_ts = pm.Normal("phi_prev_ts", mu=0, sigma=1)
    beta_age = pm.Normal("beta_age", mu=0, sigma=1)
    beta_age_sq = pm.Normal("beta_age_sq", mu=0, sigma=1)
    beta_usg = pm.Normal("beta_usg", mu=0, sigma=1)
    sigma = pm.HalfNormal("sigma", sigma=0.2)

    mu = (
        alpha
        + phi_prev_ts * prev_z
        + beta_age * age_z
        + beta_age_sq * age_z**2
        + beta_usg * usg_z
    )
    pm.Normal("ts_pct_obs", mu=mu, sigma=sigma, observed=y)

    ar_idata = pm.sample(2000, tune=2000, chains=4, target_accept=0.9, random_seed=42)
    ar_idata.extend(pm.sample_posterior_predictive(ar_idata, random_seed=42))
    pm.compute_log_likelihood(ar_idata)

In [ ]:
AR_VAR_NAMES = ["alpha", "phi_prev_ts", "beta_age", "beta_age_sq", "beta_usg", "sigma"]

ar_summary = az.summary(ar_idata, var_names=AR_VAR_NAMES, round_to=4)
print(ar_summary)

n_divergent = int(ar_idata.sample_stats["diverging"].sum())
n_draws = ar_idata.sample_stats.sizes["chain"] * ar_idata.sample_stats.sizes["draw"]
print(f"\nDivergent transitions: {n_divergent} / {n_draws}")
print(f"Max R-hat: {ar_summary['r_hat'].max():.4f} (should be close to 1.00)")
print(f"Min ESS (bulk): {ar_summary['ess_bulk'].min():.0f}")
print(f"Min ESS (tail): {ar_summary['ess_tail'].min():.0f}")

In [ ]:
os.makedirs("bayesian_ar_output", exist_ok=True)

az.plot_trace(ar_idata, var_names=AR_VAR_NAMES, compact=True)
plt.tight_layout()
plt.savefig("bayesian_ar_output/trace_plots.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Rank plots: a healthy chain mix looks roughly uniform across ranks, with no chain
# systematically higher or lower than the others
az.plot_rank(ar_idata, var_names=AR_VAR_NAMES)
plt.tight_layout()
plt.savefig("bayesian_ar_output/rank_plots.png", dpi=150)
plt.show()

# Energy plot: a healthy NUTS run has the marginal and transition energy
# distributions overlapping closely
az.plot_energy(ar_idata)
plt.tight_layout()
plt.savefig("bayesian_ar_output/energy_plot.png", dpi=150)
plt.show()

az.plot_forest(ar_idata, var_names=AR_VAR_NAMES, combined=True, hdi_prob=0.94)
plt.tight_layout()
plt.savefig("bayesian_ar_output/forest_plot.png", dpi=150)
plt.show()

In [ ]:
# Posterior predictive check: the observed TS% distribution should sit
# comfortably inside the spread of the posterior predictive draws
az.plot_ppc(ar_idata, num_pp_samples=200)
plt.tight_layout()
plt.savefig("bayesian_ar_output/posterior_predictive_check.png", dpi=150)
plt.show()

pp_samples = ar_idata.posterior_predictive["ts_pct_obs"].values.reshape(-1, len(y))
bayes_r2 = az.r2_score(y, pp_samples)
print(f"Bayesian R^2: {bayes_r2['r2']:.3f} (sd {bayes_r2['r2_std']:.3f})")

# LOO-CV: Pareto k values above ~0.7 flag observations the model struggles to predict
loo = az.loo(ar_idata, pointwise=True)
print(f"\n{loo}")

az.plot_khat(loo)
plt.tight_layout()
plt.savefig("bayesian_ar_output/pareto_k_diagnostic.png", dpi=150)
plt.show()

In [ ]:
pred_mean = ar_idata.posterior_predictive["ts_pct_obs"].mean(dim=("chain", "draw")).values

plt.figure(figsize=(6, 6))
plt.scatter(y, pred_mean, alpha=0.4, s=15)
lims = [min(y.min(), pred_mean.min()), max(y.max(), pred_mean.max())]
plt.plot(lims, lims, "r--", linewidth=1)
plt.xlabel("Observed TS%")
plt.ylabel("Posterior predictive mean TS%")
plt.title("Observed vs. predicted TS%")
plt.tight_layout()
plt.savefig("bayesian_ar_output/observed_vs_predicted.png", dpi=150)
plt.show()

In [ ]:
def project_ts_pct(age_value, prev_ts_pct_value, usg_pct_value, idata=ar_idata):
    """Posterior samples of projected TS% for a given age, prior TS%, and usage rate."""
    post = idata.posterior
    age_z_ = (age_value - age_mean) / age_sd
    usg_z_ = (usg_pct_value - usg_mean) / usg_sd
    prev_z_ = (prev_ts_pct_value - prev_mean) / prev_sd

    mu_samples = (
        post["alpha"]
        + post["phi_prev_ts"] * prev_z_
        + post["beta_age"] * age_z_
        + post["beta_age_sq"] * age_z_**2
        + post["beta_usg"] * usg_z_
    ).values.flatten()
    return mu_samples


example_samples = project_ts_pct(age_value=27, prev_ts_pct_value=0.58, usg_pct_value=0.24)
example_hdi = az.hdi(example_samples, hdi_prob=0.94)
print("Projected TS% for a 27-year-old with a 58% prior-season TS% and 24% usage:")
print(f"  mean = {example_samples.mean():.3f}, 94% HDI = [{example_hdi[0]:.3f}, {example_hdi[1]:.3f}]")

### Reading the diagnostics

- **Convergence** — `r_hat` should be at (or very close to) 1.00 and ESS
  (bulk/tail) should be in the low thousands or better; either one off
  signals the chains haven't mixed and `ar_idata` shouldn't be trusted.
- **Divergences** — any divergent transitions point to regions of the
  posterior NUTS couldn't sample reliably (often a sign priors/parameterization
  need reworking), independent of what `r_hat`/ESS say.
- **Trace/rank/energy plots** — traces should look like stationary "fuzzy
  caterpillars" with chains overlapping; rank plots should be roughly
  uniform; the energy plot's marginal and transition distributions should
  overlap closely.
- **Posterior predictive check / Bayesian R²** — how well the fitted model
  reproduces the observed TS% distribution and values.
- **Pareto k (LOO-CV)** — flags individual player-seasons the model can't
  predict well out-of-sample; a cluster of high-k points usually means a
  missing covariate or a form the Gaussian likelihood doesn't capture.

**Note on the data window:** the scrape above now covers seasons 2021-22
through 2025-26 (five completed seasons), so most players contribute up to
four lag-1 transitions instead of one. That gives `ar_model_df` repeated
observations per player, which is worth knowing about: this section still
fits one pooled regression across all player-seasons, so it doesn't yet
credit or penalize any player individually for being a persistent over/under
performer relative to the fitted curve. The EDA's variance-decomposition
check above is the one to revisit here — if it shows a large between-player
variance share, that's the signal a hierarchical model with player-level
intercepts (partial pooling) would fit meaningfully better than this pooled
version, and is now feasible with multiple observations per player.

## Hierarchical extension: player-level intercepts

The EDA's variance-decomposition check flagged a large between-player
variance share as evidence for player-level random effects, but with only
one lag-1 transition per player that structure wasn't identifiable. Now
that 2021-22 through 2025-26 are scraped, most players contribute several
transitions, so this section adds a **non-centered partial-pooling
intercept per player** on top of the pooled AR(1) model above:

```
TS%ᵢ,ₜ ~ Normal(αᵢ + φ·PREV_TS_PCT + β_age·age + β_age²·age² + β_usg·usg, σ)
αᵢ = μ_α + τ_α · offsetᵢ,   offsetᵢ ~ Normal(0, 1)
```

`μ_α` and `τ_α` are the population-level mean intercept and the
between-player spread around it — the same non-centered trick used to
avoid Neal's-funnel divergences in any hierarchical model. This section
doesn't assume the extra complexity is worth it: it fits the hierarchical
model, then compares it against the pooled model above with LOO-CV rather
than taking partial pooling on faith.

In [ ]:
player_codes, player_uniques = pd.factorize(ar_model_df["PLAYER_ID"])
player_uniques = np.asarray(player_uniques)
n_players = len(player_uniques)

print(f"{n_players} unique players across {len(ar_model_df)} player-seasons")

hier_coords = {"player": player_uniques, "obs": ar_model_df.index}

In [ ]:
with pm.Model(coords=hier_coords) as ts_ar_hier_model:
    player_idx = pm.Data("player_idx", player_codes, dims="obs")

    mu_alpha = pm.Normal("mu_alpha", mu=y.mean(), sigma=0.2)
    tau_alpha = pm.HalfNormal("tau_alpha", sigma=0.1)
    alpha_offset = pm.Normal("alpha_offset", mu=0, sigma=1, dims="player")
    alpha_player = pm.Deterministic("alpha_player", mu_alpha + tau_alpha * alpha_offset, dims="player")

    phi_prev_ts = pm.Normal("phi_prev_ts", mu=0, sigma=1)
    beta_age = pm.Normal("beta_age", mu=0, sigma=1)
    beta_age_sq = pm.Normal("beta_age_sq", mu=0, sigma=1)
    beta_usg = pm.Normal("beta_usg", mu=0, sigma=1)
    sigma = pm.HalfNormal("sigma", sigma=0.2)

    mu = (
        alpha_player[player_idx]
        + phi_prev_ts * prev_z
        + beta_age * age_z
        + beta_age_sq * age_z**2
        + beta_usg * usg_z
    )
    pm.Normal("ts_pct_obs", mu=mu, sigma=sigma, dims="obs", observed=y)

    # Hierarchical models need a higher target_accept than the pooled model
    # to avoid divergences from the non-centered funnel geometry
    hier_idata = pm.sample(2000, tune=2000, chains=4, target_accept=0.95, random_seed=42)
    hier_idata.extend(pm.sample_posterior_predictive(hier_idata, random_seed=42))
    pm.compute_log_likelihood(hier_idata)

In [ ]:
HIER_POP_VAR_NAMES = ["mu_alpha", "tau_alpha", "phi_prev_ts", "beta_age", "beta_age_sq", "beta_usg", "sigma"]

hier_summary = az.summary(hier_idata, var_names=HIER_POP_VAR_NAMES, round_to=4)
print(hier_summary)

n_divergent_hier = int(hier_idata.sample_stats["diverging"].sum())
n_draws_hier = hier_idata.sample_stats.sizes["chain"] * hier_idata.sample_stats.sizes["draw"]
print(f"\nDivergent transitions: {n_divergent_hier} / {n_draws_hier}")
print(f"Max R-hat: {hier_summary['r_hat'].max():.4f} (should be close to 1.00)")
print(f"Min ESS (bulk): {hier_summary['ess_bulk'].min():.0f}")
print(f"Min ESS (tail): {hier_summary['ess_tail'].min():.0f}")

# Per-player intercepts are checked in aggregate rather than printed individually --
# there's one for every player, so a table of 300+ rows isn't useful here
player_summary = az.summary(hier_idata, var_names=["alpha_player"], round_to=4)
print(f"\nPer-player intercepts: {len(player_summary)} players")
print(f"Max R-hat across player intercepts: {player_summary['r_hat'].max():.4f}")
print(f"Min ESS (bulk) across player intercepts: {player_summary['ess_bulk'].min():.0f}")

In [ ]:
az.plot_trace(hier_idata, var_names=HIER_POP_VAR_NAMES, compact=True)
plt.tight_layout()
plt.savefig("bayesian_ar_output/hier_trace_plots.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
az.plot_rank(hier_idata, var_names=HIER_POP_VAR_NAMES)
plt.tight_layout()
plt.savefig("bayesian_ar_output/hier_rank_plots.png", dpi=150)
plt.show()

az.plot_energy(hier_idata)
plt.tight_layout()
plt.savefig("bayesian_ar_output/hier_energy_plot.png", dpi=150)
plt.show()

In [ ]:
az.plot_ppc(hier_idata, num_pp_samples=200)
plt.tight_layout()
plt.savefig("bayesian_ar_output/hier_posterior_predictive_check.png", dpi=150)
plt.show()

hier_pp_samples = hier_idata.posterior_predictive["ts_pct_obs"].values.reshape(-1, len(y))
hier_bayes_r2 = az.r2_score(y, hier_pp_samples)
print(f"Bayesian R^2 (hierarchical): {hier_bayes_r2['r2']:.3f} (sd {hier_bayes_r2['r2_std']:.3f})")

hier_loo = az.loo(hier_idata, pointwise=True)
print(f"\n{hier_loo}")

az.plot_khat(hier_loo)
plt.tight_layout()
plt.savefig("bayesian_ar_output/hier_pareto_k_diagnostic.png", dpi=150)
plt.show()

In [ ]:
# The deciding comparison: does partial pooling actually improve out-of-sample
# fit over the simpler pooled model, or just add unneeded complexity?
model_comparison = az.compare({"pooled": ar_idata, "hierarchical": hier_idata})
print(model_comparison)

az.plot_compare(model_comparison)
plt.tight_layout()
plt.savefig("bayesian_ar_output/model_comparison.png", dpi=150)
plt.show()

In [ ]:
player_obs_counts = ar_model_df.groupby("PLAYER_ID").size()
player_emp_mean = ar_model_df.groupby("PLAYER_ID")["TS_PCT"].mean()
player_post_mean = pd.Series(
    hier_idata.posterior["alpha_player"].mean(dim=("chain", "draw")).values,
    index=player_uniques,
)

shrinkage_df = pd.DataFrame({
    "n_obs": player_obs_counts,
    "empirical_mean_ts": player_emp_mean,
    "posterior_mean_intercept": player_post_mean,
}).dropna()

plt.figure(figsize=(7, 6))
sc = plt.scatter(
    shrinkage_df["empirical_mean_ts"],
    shrinkage_df["posterior_mean_intercept"],
    c=shrinkage_df["n_obs"],
    cmap="viridis",
    alpha=0.7,
    s=25,
)
lims = [
    shrinkage_df[["empirical_mean_ts", "posterior_mean_intercept"]].min().min(),
    shrinkage_df[["empirical_mean_ts", "posterior_mean_intercept"]].max().max(),
]
plt.plot(lims, lims, "r--", linewidth=1, label="no shrinkage")
plt.colorbar(sc, label="player-seasons observed")
plt.xlabel("Empirical mean TS% (raw)")
plt.ylabel("Posterior mean player intercept")
plt.title("Partial-pooling shrinkage toward the population mean")
plt.legend()
plt.tight_layout()
plt.savefig("bayesian_ar_output/shrinkage_plot.png", dpi=150)
plt.show()

In [ ]:
def project_ts_pct_hier(age_value, prev_ts_pct_value, usg_pct_value, player_id=None, idata=hier_idata):
    """Posterior samples of projected TS%, using a player's own partial-pooled
    intercept when player_id is a player seen in ar_model_df, otherwise drawing
    from the population-level intercept distribution (e.g. an incoming rookie)."""
    post = idata.posterior
    age_z_ = (age_value - age_mean) / age_sd
    usg_z_ = (usg_pct_value - usg_mean) / usg_sd
    prev_z_ = (prev_ts_pct_value - prev_mean) / prev_sd

    if player_id is not None and player_id in player_uniques:
        p_idx = int(np.where(player_uniques == player_id)[0][0])
        alpha_samples = post["alpha_player"].isel(player=p_idx)
    else:
        rng = np.random.default_rng(0)
        alpha_samples = post["mu_alpha"] + post["tau_alpha"] * rng.standard_normal(post["mu_alpha"].shape)

    mu_samples = (
        alpha_samples
        + post["phi_prev_ts"] * prev_z_
        + post["beta_age"] * age_z_
        + post["beta_age_sq"] * age_z_**2
        + post["beta_usg"] * usg_z_
    ).values.flatten()
    return mu_samples


example_player = ar_model_df["PLAYER_ID"].iloc[0]
hier_samples = project_ts_pct_hier(age_value=27, prev_ts_pct_value=0.58, usg_pct_value=0.24, player_id=example_player)
hier_hdi = az.hdi(hier_samples, hdi_prob=0.94)
print(f"Projected TS% for player {example_player} (partial-pooled intercept):")
print(f"  mean = {hier_samples.mean():.3f}, 94% HDI = [{hier_hdi[0]:.3f}, {hier_hdi[1]:.3f}]")

new_player_samples = project_ts_pct_hier(age_value=27, prev_ts_pct_value=0.58, usg_pct_value=0.24, player_id=None)
new_player_hdi = az.hdi(new_player_samples, hdi_prob=0.94)
print("\nProjected TS% for a player not in the training data (population-level intercept):")
print(f"  mean = {new_player_samples.mean():.3f}, 94% HDI = [{new_player_hdi[0]:.3f}, {new_player_hdi[1]:.3f}]")

### Reading the hierarchical extension

- **`mu_alpha` / `tau_alpha`** — the population-level mean intercept and the
  between-player standard deviation around it. A `tau_alpha` posterior
  concentrated near zero means the data don't support much player-to-player
  variation beyond what `PREV_TS_PCT`, age, and usage already explain — the
  AR(1) term is itself a strong proxy for "who this player is," so don't
  expect this to match the EDA's raw between-player variance share, which
  was computed without conditioning on the prior season.
- **Per-player intercept diagnostics** — checked in aggregate (max r-hat,
  min ESS) rather than printed individually. A handful of players near a
  divergence or with low ESS is normal (usually a player with very few
  observations); a widespread pattern signals a convergence issue with the
  whole hierarchy, not just those players.
- **Model comparison (`az.compare`)** — the deciding evidence for whether
  the added complexity is worth it. Higher `elpd_loo` favors the
  hierarchical model; if the pooled model comes out on top, or the two are
  statistically indistinguishable given `dse`, prefer the simpler pooled
  model from the section above.
- **Shrinkage plot** — points near the red no-shrinkage line have enough of
  their own data that the model trusts their empirical mean; points pulled
  toward the population mean (especially low-observation-count ones) are
  exactly the partial-pooling behavior a hierarchical model is for — a
  rookie or injury-shortened player shouldn't be judged purely on a handful
  of noisy games.
- **New-player projections** — `project_ts_pct_hier` falls back to sampling
  from the population-level intercept distribution for any `player_id` not
  in `ar_model_df`, which is the right way to project a player the model
  has never seen (e.g., an incoming rookie) rather than reusing another
  player's fitted intercept.